In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import h5py
import matplotlib.pyplot as plt
import random
import os
from torchvision.ops import box_iou
from tqdm import tqdm

In [2]:
import hdf5plugin

In [3]:
# Utilities

In [ ]:

def create_voxel_grid(events, height=480, width=640, num_bins=10):
    """
    Convert event stream into a voxel grid representation.
    
    Args:
        events: Dictionary containing event data ('t', 'x', 'y', 'p')
        height: Image height
        width: Image width
        num_bins: Number of time bins for the voxel grid
        
    Returns:
        torch.Tensor: Voxel grid of shape (num_bins, height, width)
    """
    voxel_grid = np.zeros((num_bins, height, width), dtype=np.float32)
    
    if len(events["t"]) == 0:
        return torch.tensor(voxel_grid, dtype=torch.float32)
    
    t_min, t_max = events["t"].min(), events["t"].max()
    t_range = t_max - t_min + 1e-6  
    
    # Assign each event to a time bin
    bin_indices = ((events["t"] - t_min) / t_range * num_bins).astype(np.int32)
    bin_indices = np.clip(bin_indices, 0, num_bins - 1)
    
    # Accumulate events 
    for i in range(len(events["x"])):
        x, y, b, p = events["x"][i], events["y"][i], bin_indices[i], events["p"][i]

        if 0 <= x < width and 0 <= y < height:

            voxel_grid[b, y, x] += 1 if p > 0 else -1
    
    # Normalize
    if np.max(np.abs(voxel_grid)) > 0:
        voxel_grid = voxel_grid / np.max(np.abs(voxel_grid))
    
    return torch.tensor(voxel_grid, dtype=torch.float32)
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

def visualize_voxel_grid(voxel_grid, labels=None, save_path=None, detection_map=None):

    if len(voxel_grid.shape) == 4:

        voxel_grid = voxel_grid[0]
    

    fig = plt.figure(figsize=(18, 6))
    
    # Plot 1: Event frame with bounding boxes
    plt.subplot(1, 2 if detection_map is not None else 1, 1)
    
    # Aggregate across time bins
    event_frame = voxel_grid.sum(dim=0).numpy()
    
    # Normalize
    event_frame = (event_frame - event_frame.min()) / (event_frame.max() - event_frame.min() + 1e-6)
    
    plt.imshow(event_frame, cmap="gray")
    
    # Overlay bounding boxes
    if labels is not None:
        for i, det in enumerate(labels):
            color = 'red' if i == 0 else 'lime'  # Ground truth in red, prediction in green
            label_text = "Ground Truth" if i == 0 else "Prediction"
            
            x, y, w, h = det.tolist()
            rect = plt.Rectangle((x, y), w, h, fill=False, edgecolor=color, linewidth=2)
            plt.gca().add_patch(rect)
            plt.text(x, y - 5, label_text, color=color, fontsize=10,
                     bbox=dict(facecolor="black", alpha=0.5))
    
    plt.title("Event Frame with Bounding Boxes")
    plt.axis("off")
    
    # Plot 2: Confidence map
    if detection_map is not None:
        plt.subplot(1, 2, 2)
        
        # confidence map
        conf_map = torch.sigmoid(detection_map[4]).cpu().numpy()
        
        plt.imshow(conf_map, cmap="plasma", vmin=0, vmax=1)
        plt.colorbar(label="Confidence")
        plt.title("Detection Confidence Map")
        plt.axis("off")
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path)
        plt.close()
    else:
        plt.show()

def calculate_iou(box1, box2):
    """
    Calculate IoU between two boxes in format [x, y, w, h]
    """
    # Convert to [x1, y1, x2, y2] format
    box1_x1, box1_y1 = box1[0], box1[1]
    box1_x2, box1_y2 = box1[0] + box1[2], box1[1] + box1[3]
    
    box2_x1, box2_y1 = box2[0], box2[1]
    box2_x2, box2_y2 = box2[0] + box2[2], box2[1] + box2[3]
    
    # Calculate intersection area
    x_left = max(box1_x1, box2_x1)
    y_top = max(box1_y1, box2_y1)
    x_right = min(box1_x2, box2_x2)
    y_bottom = min(box1_y2, box2_y2)
    
    if x_right < x_left or y_bottom < y_top:
        return 0.0
    
    intersection_area = (x_right - x_left) * (y_bottom - y_top)
    
    # Calculate union area
    box1_area = (box1_x2 - box1_x1) * (box1_y2 - box1_y1)
    box2_area = (box2_x2 - box2_x1) * (box2_y2 - box2_y1)
    union_area = box1_area + box2_area - intersection_area
    
    if union_area <= 0:
        return 0.0
        
    return intersection_area / union_area

def calculate_metrics(pred_boxes, gt_boxes, iou_thresholds=[0.5]):
    """
    Calculate detection metrics (AP) at different IoU thresholds.
    """
    metrics = {}
    batch_size = pred_boxes.size(0)
    
    # Calculate IoU
    ious = torch.zeros(batch_size)
    for i in range(batch_size):
        # Calculate IoU between prediction and ground truth
        iou = calculate_iou(pred_boxes[i].cpu(), gt_boxes[i].cpu())
        ious[i] = iou
    
    # Calculate AP at different IoUs
    for threshold in iou_thresholds:
        matches = (ious > threshold).float()
        precision = matches.mean().item()
        metrics[f'AP@{threshold}'] = precision
    
    # Calculate mean IoU
    metrics['Mean_IoU'] = ious.mean().item()
    
    return metrics

def debug_predictions(model, dataloader, device, num_samples=3):
    """Debug model predictions on a few samples"""
    model.eval()
    examples = []
    
    with torch.no_grad():
        for batch in dataloader:
            if batch is None or len(examples) >= num_samples:
                continue
                
            voxel_grids, bboxes, _ = batch
            voxel_grids = voxel_grids.to(device)
            
            bbox_pred, conf_pred, detection_maps = model(voxel_grids)
            
            for i in range(min(len(voxel_grids), num_samples - len(examples))):
                examples.append({
                    'voxel_grid': voxel_grids[i].cpu(),
                    'gt_bbox': bboxes[i].cpu(),
                    'pred_bbox': bbox_pred[i].cpu(),
                    'confidence': conf_pred[i].item(),
                    'detection_map': detection_maps[i].cpu()
                })
                
            if len(examples) >= num_samples:
                break
    
    return examples


class BBoxLoss(nn.Module):
    """
    Custom loss function for bounding box regression."""
    def __init__(self, img_width=640, img_height=480):
        super(BBoxLoss, self).__init__()
        self.img_width = img_width
        self.img_height = img_height
        self.mse_loss = nn.MSELoss(reduction='mean')
        self.smooth_l1 = nn.SmoothL1Loss(reduction='mean')
    
    def forward(self, pred, target):
        """
        Compute box regression loss
        Args:
            pred: Predicted boxes in format [x, y, w, h]
            target: Target boxes in same format
        """
        # Compute MSE loss for xy coordinates
        coord_loss = self.mse_loss(pred[:, :2], target[:, :2])
        
        # Compute Smooth L1 loss for width/height (handles small boxes better)
        size_loss = self.smooth_l1(pred[:, 2:], target[:, 2:])
        
        # Combined loss - weighting coordinate matching higher
        return coord_loss * 2.0 + size_loss


In [5]:
#BaseLine CNN

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class BaselineCNN(nn.Module):
    def __init__(self, num_bins=5, num_classes=1, img_height=480, img_width=640):
        super(BaselineCNN, self).__init__()
        
        self.num_classes = num_classes
        self.img_height = img_height
        self.img_width = img_width
        

        self.feature_extractor = nn.Sequential(
            # First conv block
            nn.Conv2d(num_bins, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),  # Output: 16 x 240 x 320
            
            # Second conv block
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),  # Output: 32 x 120 x 160
            
            # Third conv block
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),  # Output: 64 x 60 x 80
        )
        
        # Detection head
        self.detection_head = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 5, kernel_size=1)  # 5 channels: 4 for bbox + 1 for confidence
        )
        
        # Initialize weights, improve strat
        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                # Use Xavier/Glorot
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
    
    def forward(self, x):
        """
        Forward pass with spatial detection approach
        """
        batch_size = x.size(0)
        
        # Extract features while keeping spatial information
        features = self.feature_extractor(x)
        

        detection_maps = self.detection_head(features)  # B x 5 x H x W
        
        # Get confidence map
        conf_map = torch.sigmoid(detection_maps[:, 4:5])  # B x 1 x H x W
        
        # Find max confidence locations
        B, _, H, W = conf_map.shape
        flat_conf = conf_map.view(B, -1)
        max_conf_idx = torch.argmax(flat_conf, dim=1)
        
        # Convert flat indices to 2D coordinates
        y_indices = max_conf_idx // W
        x_indices = max_conf_idx % W
        
        # Collect bbox predictions at those locations
        bbox_preds = []
        conf_preds = []
        
        for b in range(B):
            y_idx = y_indices[b]
            x_idx = x_indices[b]
            
            # Get raw bbox values at max confidence location
            raw_bbox = detection_maps[b, :4, y_idx, x_idx]
            

            scale_x = self.img_width / W
            scale_y = self.img_height / H
            
            # Center coordinates in feature map space
            cx = x_idx.float() + 0.5  # +0.5 to get center of cell
            cy = y_idx.float() + 0.5
            
            # Center coordinates in image space
            cx_img = cx * scale_x
            cy_img = cy * scale_y
            

            w_img = torch.sigmoid(raw_bbox[2]) * self.img_width * 0.5
            h_img = torch.sigmoid(raw_bbox[3]) * self.img_height * 0.5
            

            dx = torch.tanh(raw_bbox[0]) * scale_x * 2  # Allow ±2 cell offsets
            dy = torch.tanh(raw_bbox[1]) * scale_y * 2
            
            # Final bbox coordinates
            x_img = cx_img + dx - w_img/2  # Convert center-x to top-left-x
            y_img = cy_img + dy - h_img/2  # Convert center-y to top-left-y
            
            # Clip
            x_img = torch.clamp(x_img, 0, self.img_width - w_img)
            y_img = torch.clamp(y_img, 0, self.img_height - h_img)
            w_img = torch.clamp(w_img, 5, self.img_width)  # Minimum width 5px
            h_img = torch.clamp(h_img, 5, self.img_height)  # Minimum height 5px
            

            bbox_pred = torch.stack([x_img, y_img, w_img, h_img])
            bbox_preds.append(bbox_pred)
            

            conf_pred = conf_map[b, 0, y_idx, x_idx]
            conf_preds.append(conf_pred)
        
        # Stack predictions
        bbox_preds = torch.stack(bbox_preds)
        conf_preds = torch.stack(conf_preds).unsqueeze(1)
        
        return bbox_preds, conf_preds, detection_maps

In [7]:

class SpatialDetectionLoss(nn.Module):
    """
    Combined loss function for spatial detection approach
    """
    def __init__(self, img_width=640, img_height=480, lambda_coord=5.0, lambda_noobj=0.5):
        super(SpatialDetectionLoss, self).__init__()
        self.img_width = img_width
        self.img_height = img_height
        self.lambda_coord = lambda_coord  # Weight for bbox coordinate loss
        self.lambda_noobj = lambda_noobj  # Weight for no-object confidence loss
        self.mse_loss = nn.MSELoss(reduction='mean')
        self.bce_loss = nn.BCELoss(reduction='mean')
        
    def forward(self, predictions, targets):
        """
        Compute loss for spatial detection model
        
        Args:
            predictions: Tuple of (bbox_pred, conf_pred, detection_maps)
                bbox_pred: Tensor of shape (B, 4) with [x, y, w, h]
                conf_pred: Tensor of shape (B, 1) with confidence scores
                detection_maps: Tensor of shape (B, 5, H, W) with detection maps
            targets: Tensor of shape (B, 4) with ground truth boxes
        
        Returns:
            total_loss: Combined loss value
            loss_components: Dict with individual loss components
        """
        bbox_pred, conf_pred, detection_maps = predictions
        batch_size = bbox_pred.size(0)
        
        # 1. Bbox coordinate loss (direct supervision)
        coord_loss = self.mse_loss(bbox_pred, targets)
        
        # 2. Create target confidence map by placing gaussians at ground truth locations
        B, _, H, W = detection_maps.shape
        confidence_targets = torch.zeros_like(detection_maps[:, 4])
        
        for b in range(batch_size):
            # Get ground truth in normalized coordinates
            x, y, w, h = targets[b]
            
            # Convert to feature map coordinates
            fm_x = x * W / self.img_width
            fm_y = y * H / self.img_height
            fm_w = w * W / self.img_width
            fm_h = h * H / self.img_height
            
            # Get center point
            fm_cx = fm_x + fm_w / 2
            fm_cy = fm_y + fm_h / 2
            
            # Place a gaussian at center point
            x_coords = torch.arange(0, W, device=targets.device).float()
            y_coords = torch.arange(0, H, device=targets.device).float()
            y_grid, x_grid = torch.meshgrid(y_coords, x_coords, indexing='ij')
            
            # Gaussian falloff based on distance to center and bbox size
            sigma_x = max(fm_w / 2, 1.0)
            sigma_y = max(fm_h / 2, 1.0)
            
            gaussian = torch.exp(
                -0.5 * (((x_grid - fm_cx) / sigma_x) ** 2 + ((y_grid - fm_cy) / sigma_y) ** 2)
            )
            
            # Set the confidence target
            confidence_targets[b] = gaussian
        
        # 3. Confidence loss (BCE with target map)
        conf_map_pred = torch.sigmoid(detection_maps[:, 4])
        
        # Positive samples (where target > 0.5)
        pos_mask = confidence_targets > 0.5
        if pos_mask.sum() > 0:
            pos_loss = F.binary_cross_entropy_with_logits(
                detection_maps[:, 4][pos_mask], 
                confidence_targets[pos_mask],
                reduction='mean'
            )
        else:
            pos_loss = torch.tensor(0.0, device=targets.device)
        
        # Negative samples (hard negative mining: take top k hardest negatives)
        neg_mask = confidence_targets <= 0.5
        k = 3 * pos_mask.sum().item()  # 3x as many negatives as positives
        k = max(k, batch_size)  # At least batch_size negatives
        k = min(k, neg_mask.sum().item())  # But not more than available negatives
        
        if k > 0 and neg_mask.sum() > 0:
            # Get hardest negative examples (highest confidence where target is low)
            neg_conf = detection_maps[:, 4][neg_mask]
            neg_sorted, _ = torch.sort(neg_conf, descending=True)
            threshold = neg_sorted[min(k, len(neg_sorted) - 1)]
            hard_neg_mask = (detection_maps[:, 4] > threshold) & neg_mask
            
            neg_loss = F.binary_cross_entropy_with_logits(
                detection_maps[:, 4][hard_neg_mask],
                confidence_targets[hard_neg_mask],
                reduction='mean'
            )
        else:
            neg_loss = torch.tensor(0.0, device=targets.device)
        
        # Direct confidence for the predictions (should be high)
        direct_conf_loss = F.binary_cross_entropy(conf_pred, torch.ones_like(conf_pred))
        
        # Combine losses
        conf_loss = pos_loss + self.lambda_noobj * neg_loss + direct_conf_loss
        total_loss = self.lambda_coord * coord_loss + conf_loss
        
        # Return individual components for monitoring
        loss_components = {
            'coord_loss': coord_loss.item(),
            'conf_pos_loss': pos_loss.item() if not isinstance(pos_loss, float) else pos_loss,
            'conf_neg_loss': neg_loss.item() if not isinstance(neg_loss, float) else neg_loss,
            'direct_conf_loss': direct_conf_loss.item(),
            'total_loss': total_loss.item()
        }
        
        return total_loss, loss_components

In [8]:
#Dataset & Dataloader

In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
import os
import time
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
import random


In [ ]:
import torch
import numpy as np
import h5py
from torch.utils.data import Dataset
import os
from tqdm import tqdm
import random
import multiprocessing as mp
from torch.utils.data.dataloader import default_collate

def create_voxel_grid(events, height=480, width=640, num_bins=10):
    """
    Convert event stream into a voxel grid representation - optimized version.
    """
    # Early return if no events
    if len(events["t"]) == 0:
        return torch.zeros((num_bins, height, width), dtype=torch.float32)
    
    voxel_grid = np.zeros((num_bins, height, width), dtype=np.float32)
    
    t_min, t_max = events["t"].min(), events["t"].max()
    t_range = t_max - t_min + 1e-6  # Prevent division by zero
    
    # Vectorized bin assignment
    bin_indices = np.clip(((events["t"] - t_min) / t_range * num_bins).astype(np.int32), 0, num_bins - 1)
    
    # Filter coordinates that are out of bounds (vectorized)
    valid_indices = (events["x"] >= 0) & (events["x"] < width) & (events["y"] >= 0) & (events["y"] < height)
    x = events["x"][valid_indices]
    y = events["y"][valid_indices]
    b = bin_indices[valid_indices]
    p = events["p"][valid_indices]
    
    # Use numpy's advanced indexing for faster assignment
    for i in range(len(x)):
        voxel_grid[b[i], y[i], x[i]] += 1 if p[i] > 0 else -1
    
    # Quick normalization
    if np.max(np.abs(voxel_grid)) > 0:
        voxel_grid = voxel_grid / np.max(np.abs(voxel_grid))
    
    return torch.tensor(voxel_grid, dtype=torch.float32)

class PrecomputedEventDataset(Dataset):
    """
    Optimized dataset that precomputes voxel grids during initialization
    to avoid repeated computations during training.
    """
    def __init__(self, event_files, label_files, num_bins=10, time_window=20000, 
                 transform=None, frame_step=100, max_samples_per_file=None,
                 precompute=True, cache_dir=None):
        """
        Dataset for event-based object detection with precomputation.
        """
        self.event_files = event_files
        self.label_files = label_files
        self.num_bins = num_bins
        self.time_window = time_window
        self.transform = transform
        self.max_events = 5000  # Reduced max events for faster processing
        self.min_events = 100
        self.frame_step = frame_step
        self.max_samples_per_file = max_samples_per_file
        self.precompute = precompute
        self.cache_dir = cache_dir
        
        if self.cache_dir and not os.path.exists(self.cache_dir):
            os.makedirs(self.cache_dir)
        
        # Load labels and correct timestamps
        self.all_samples = []
        
        print("Loading and processing data...")
        for file_idx, (event_file, label_file) in enumerate(tqdm(zip(event_files, label_files), total=len(event_files))):
            try:
                # Load labels
                labels = np.load(label_file, allow_pickle=True)
                
                # Get t_offset
                with h5py.File(event_file, "r", swmr=True) as f:
                    t_offset = f.get("t_offset", 0)[()]
                    # Get all timestamps at once
                    events_t = f["events/t"][:]
                
                # Correct timestamps
                labels["t"] -= t_offset
                
                # Select samples with frame_step
                if self.max_samples_per_file:
                    indices = list(range(0, len(labels), self.frame_step))
                    if len(indices) > self.max_samples_per_file:
                        indices = random.sample(indices, self.max_samples_per_file)
                else:
                    indices = range(0, len(labels), self.frame_step)
                
                # Filter and add valid samples
                for label_idx in indices:
                    if label_idx >= len(labels):
                        continue
                        
                    label = labels[label_idx]
                    timestamp = label["t"]
                    
                    # Find events in time window
                    event_indices = np.where(
                        (events_t >= timestamp - self.time_window) &
                        (events_t <= timestamp + self.time_window)
                    )[0]
                    
                    # Filter by event count
                    if len(event_indices) < self.min_events:
                        continue
                    
                    # Limit max events by random sampling
                    if len(event_indices) > self.max_events:
                        event_indices = np.random.choice(event_indices, self.max_events, replace=False)
                    
                    # Store sample info
                    self.all_samples.append({
                        'file_idx': file_idx,
                        'event_file': event_file,
                        'label': {k: label[k] for k in ['x', 'y', 'w', 'h', 'class_id']},
                        'timestamp': timestamp,
                        'event_indices': event_indices
                    })
            except Exception as e:
                print(f"Error processing file {event_file}: {e}")
        
        print(f"Total valid samples: {len(self.all_samples)}")
        
        # Precompute voxel grids if requested
        self.voxel_grids = {}
        if self.precompute:
            self._precompute_voxel_grids()
    
    def _precompute_voxel_grids(self):
        """Precompute voxel grids to avoid repeated computation during training."""
        print("Precomputing voxel grids...")
        
        for idx in tqdm(range(len(self.all_samples))):
            # Check if we have a cached version on disk
            if self.cache_dir:
                cache_file = os.path.join(self.cache_dir, f"voxel_grid_{idx}.pt")
                if os.path.exists(cache_file):
                    self.voxel_grids[idx] = cache_file
                    continue
            
            # Otherwise compute and store in memory
            sample = self.all_samples[idx]
            event_file = sample['event_file']
            event_indices = sample['event_indices']
            
            try:
                with h5py.File(event_file, "r", swmr=True) as f:
                    event_data = {
                        key: f["events/" + key][:][event_indices] 
                        for key in ["t", "x", "y", "p"]
                    }
                
                voxel_grid = create_voxel_grid(event_data, num_bins=self.num_bins)
                
                if self.cache_dir:
                    # Save to disk
                    cache_file = os.path.join(self.cache_dir, f"voxel_grid_{idx}.pt")
                    torch.save(voxel_grid, cache_file)
                    self.voxel_grids[idx] = cache_file
                else:
                    # Keep in memory
                    self.voxel_grids[idx] = voxel_grid
                    
            except Exception as e:
                print(f"Error precomputing voxel grid for sample {idx}: {e}")
                self.voxel_grids[idx] = torch.zeros((self.num_bins, 480, 640), dtype=torch.float32)
    
    def __len__(self):
        return len(self.all_samples)
    
    def __getitem__(self, idx):
        sample = self.all_samples[idx]
        

        if self.precompute:
            if isinstance(self.voxel_grids[idx], str):

                voxel_grid = torch.load(self.voxel_grids[idx], weights_only=True)
            else:
                # Get from memory
                voxel_grid = self.voxel_grids[idx]
        else:
            # Compute on-the-fly
            event_file = sample['event_file']
            event_indices = sample['event_indices']
            
            try:
                with h5py.File(event_file, "r", swmr=True) as f:
                    event_data = {
                        key: f["events/" + key][:][event_indices] 
                        for key in ["t", "x", "y", "p"]
                    }
                
                voxel_grid = create_voxel_grid(event_data, num_bins=self.num_bins)
                
            except Exception as e:
                print(f"Error computing voxel grid for sample {idx}: {e}")
                voxel_grid = torch.zeros((self.num_bins, 480, 640), dtype=torch.float32)
        
        # Extract label
        label = sample['label']
        bbox = torch.tensor([label["x"], label["y"], label["w"], label["h"]], dtype=torch.float32)
        
        # Apply transforms if any
        if self.transform:
            voxel_grid = self.transform(voxel_grid)
        
        return voxel_grid, bbox, label["class_id"]


def collate_fn(batch):

    batch = [b for b in batch if b is not None]
    if len(batch) == 0:
        return None
    return default_collate(batch)

In [11]:
#Train & Eval

In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer, device, clip_grad=None):
    """Train the model for one epoch and return average loss."""
    model.train()
    total_loss = 0
    loss_components = {
        'coord_loss': 0, 
        'conf_pos_loss': 0, 
        'conf_neg_loss': 0, 
        'direct_conf_loss': 0,
        'total_loss': 0
    }
    batches_processed = 0
    
    progress_bar = tqdm(dataloader, desc="Training")
    for i, batch in enumerate(progress_bar):
        if batch is None:  # Skip bad batches
            continue
            
        voxel_grids, bboxes, _ = batch
        
        # Move data to device
        voxel_grids = voxel_grids.to(device)
        bboxes = bboxes.to(device)
        
        # Forward pass
        predictions = model(voxel_grids)
        
        # Calculate loss
        loss, components = criterion(predictions, bboxes)
        

        optimizer.zero_grad()
        loss.backward()

        if clip_grad is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip_grad)
            
        optimizer.step()
        
        # Update progress bar
        total_loss += loss.item()
        batches_processed += 1
        

        for k, v in components.items():
            loss_components[k] += v
        
        # Show current loss in progress bar
        progress_bar.set_postfix({"loss": total_loss / (batches_processed)})
        

        if i % 20 == 0:
            with torch.no_grad():

                if len(predictions[0]) > 0:
                    bbox_pred, conf_pred, _ = predictions
                    print(f"\nBatch {i} - Sample 0:")
                    print(f"  GT: {bboxes[0].cpu().numpy()}")
                    print(f"  Pred: {bbox_pred[0].cpu().numpy()}")
                    print(f"  Confidence: {conf_pred[0].item():.4f}")
                    print(f"  IoU: {calculate_iou(bbox_pred[0].cpu(), bboxes[0].cpu()):.4f}")
    

    if batches_processed > 0:
        for k in loss_components:
            loss_components[k] /= batches_processed
    
    return total_loss / batches_processed if batches_processed > 0 else float('inf'), loss_components

def evaluate(model, dataloader, device, iou_thresholds=[0.5]):
    """Evaluate the model and return metrics."""
    model.eval()
    all_metrics = {f'AP@{threshold}': 0 for threshold in iou_thresholds}
    all_metrics['Mean_IoU'] = 0
    batches_processed = 0
    
    # Debug predictions
    debug_samples = []
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            if batch is None:  # Skip bad batches
                continue
                
            voxel_grids, bboxes, _ = batch
            
            # Move data to device
            voxel_grids = voxel_grids.to(device)
            bboxes = bboxes.to(device)
            
            # Forward pass
            bbox_pred, conf_pred, _ = model(voxel_grids)
            
            # Store some predictions for debugging
            if len(debug_samples) < 5:
                for i in range(min(2, len(bbox_pred))):
                    debug_samples.append({
                        'gt': bboxes[i].cpu().numpy(),
                        'pred': bbox_pred[i].cpu().numpy(),
                        'conf': conf_pred[i].item()
                    })
            
            # Calculate metrics
            metrics = calculate_metrics(bbox_pred, bboxes, iou_thresholds)
            
            # Accumulate metrics
            for key, value in metrics.items():
                all_metrics[key] += value
            
            batches_processed += 1
    

    print("\n=== Debugging Evaluation Predictions ===")
    for i, sample in enumerate(debug_samples):
        print(f"Sample {i+1} - Ground Truth: {sample['gt']}")
        print(f"Sample {i+1} - Prediction  : {sample['pred']} (Conf: {sample['conf']:.4f})")
    
    # Average metrics
    if batches_processed > 0:
        for key in all_metrics:
            all_metrics[key] /= batches_processed
    
    return all_metrics

def test_iou_calculation():
    """Test the IoU calculation with a known case."""
    # Create two boxes with known IoU
    box1 = torch.tensor([100.0, 100.0, 50.0, 50.0])  # x, y, w, h
    box2 = torch.tensor([125.0, 125.0, 50.0, 50.0])  # This should have IoU of ~0.14
    
    iou = calculate_iou(box1, box2)
    print(f"\n=== Testing IoU Computation ===")
    print(f"Expected IoU ≈ 0.14, Computed IoU: {iou}")

In [13]:
#Testing ETC

'import torch\nimport os\nimport numpy as np\nfrom torch.utils.data import DataLoader, random_split\nimport time\nfrom tqdm import tqdm\n\n# --- Assumed imports ---\n# from your_module import BaselineCNN, PrecomputedEventDataset, collate_fn, evaluate, visualize_voxel_grid, compute_iou\ndef compute_iou(box1, box2):\n    # Compute area of each box\n    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])\n    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])\n\n    # Compute intersection\n    lt = torch.max(box1[:2], box2[:2])  # Left top corner\n    rb = torch.min(box1[2:], box2[2:])  # Right bottom corner\n    wh = (rb - lt).clamp(min=0)  # Width and height of intersection\n    inter = wh[0] * wh[1]  # Intersection area\n\n    # Compute union\n    union = area1 + area2 - inter\n\n    # Compute IoU\n    iou = inter / union if union > 0 else torch.tensor(0.0)  # Avoid division by zero\n    return iou\n\n# 1. Unit test for IoU computation.\ndef test_iou():\n    print("\n=== Testing IoU Compu

In [15]:
#Main Training Script

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
import os
import time
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
import random
import pickle

##############################
# Utility Functions & Classes
##############################

def set_seed(seed=42):
    """Set seed for reproducibility"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

def visualize_voxel_grid(voxel_grid, labels=None, save_path=None, detection_map=None):

    if len(voxel_grid.shape) == 4:
        voxel_grid = voxel_grid[0]
    
    fig = plt.figure(figsize=(18, 6))
    
    # Plot event frame with bounding boxes
    plt.subplot(1, 2 if detection_map is not None else 1, 1)
    event_frame = voxel_grid.sum(dim=0).numpy()
    event_frame = (event_frame - event_frame.min()) / (event_frame.max() - event_frame.min() + 1e-6)
    plt.imshow(event_frame, cmap="gray")
    
    if labels is not None:
        for i, det in enumerate(labels):
            color = 'red' if i == 0 else 'lime'
            label_text = "Ground Truth" if i == 0 else "Prediction"
            x, y, w, h = det.tolist()
            rect = plt.Rectangle((x, y), w, h, fill=False, edgecolor=color, linewidth=2)
            plt.gca().add_patch(rect)
            plt.text(x, y - 5, label_text, color=color, fontsize=10,
                     bbox=dict(facecolor="black", alpha=0.5))
    plt.title("Event Frame with Bounding Boxes")
    plt.axis("off")
    
    if detection_map is not None:
        plt.subplot(1, 2, 2)
        conf_map = torch.sigmoid(detection_map[4]).cpu().numpy()
        plt.imshow(conf_map, cmap="plasma", vmin=0, vmax=1)
        plt.colorbar(label="Confidence")
        plt.title("Detection Confidence Map")
        plt.axis("off")
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path)
        plt.close()
    else:
        plt.show()

def calculate_iou(box1, box2):
    """
    Calculate IoU between two boxes in format [x, y, w, h]
    """
    box1_x1, box1_y1 = box1[0], box1[1]
    box1_x2, box1_y2 = box1[0] + box1[2], box1[1] + box1[3]
    box2_x1, box2_y1 = box2[0], box2[1]
    box2_x2, box2_y2 = box2[0] + box2[2], box2[1] + box2[3]
    
    x_left = max(box1_x1, box2_x1)
    y_top = max(box1_y1, box2_y1)
    x_right = min(box1_x2, box2_x2)
    y_bottom = min(box1_y2, box2_y2)
    
    if x_right < x_left or y_bottom < y_top:
        return 0.0
    
    intersection_area = (x_right - x_left) * (y_bottom - y_top)
    box1_area = (box1_x2 - box1_x1) * (box1_y2 - box1_y1)
    box2_area = (box2_x2 - box2_x1) * (box2_y2 - box2_y1)
    union_area = box1_area + box2_area - intersection_area
    
    if union_area <= 0:
        return 0.0
    return intersection_area / union_area

def calculate_metrics(pred_boxes, gt_boxes, iou_thresholds=[0.5]):

    metrics = {}
    batch_size = pred_boxes.size(0)
    ious = torch.zeros(batch_size)
    for i in range(batch_size):
        ious[i] = calculate_iou(pred_boxes[i].cpu(), gt_boxes[i].cpu())
    
    for threshold in iou_thresholds:
        matches = (ious > threshold).float()
        precision = matches.mean().item()
        metrics[f'AP@{threshold}'] = precision
    metrics['Mean_IoU'] = ious.mean().item()
    return metrics

def collate_fn(batch):
    """Simple collate function to stack samples."""
    batch = list(filter(lambda x: x is not None, batch))
    voxel_grids, bboxes, extras = zip(*batch)
    return torch.stack(voxel_grids), torch.stack(bboxes), extras

#############################################
# Dataset Class with Caching to Disk
#############################################

class PrecomputedEventDataset(torch.utils.data.Dataset):
    def __init__(self, event_files, label_files, num_bins, time_window, frame_step,
                 max_samples_per_file, precompute, cache_dir):
        self.event_files = event_files
        self.label_files = label_files
        self.num_bins = num_bins
        self.time_window = time_window
        self.frame_step = frame_step
        self.max_samples_per_file = max_samples_per_file
        self.precompute = precompute
        self.cache_dir = cache_dir
        self.samples = []
        
        os.makedirs(self.cache_dir, exist_ok=True)
        
        for efile, lfile in zip(self.event_files, self.label_files):
            cache_file = os.path.join(self.cache_dir, os.path.basename(efile) + ".pkl")
            if os.path.exists(cache_file):
                print(f"Loading cached data for {efile}")
                with open(cache_file, "rb") as f:
                    samples = pickle.load(f)
            else:
                print(f"Processing data for {efile}")
                samples = self.process_file(efile, lfile)
                with open(cache_file, "wb") as f:
                    pickle.dump(samples, f)
            self.samples.extend(samples)
    
    def process_file(self, event_file, label_file):
        """
        Dummy processing: replace with your actual logic.
        For demonstration, we create max_samples_per_file samples with random data.
        """
        samples = []
        for i in range(self.max_samples_per_file):
            # Create a dummy voxel grid of shape [num_bins, 480, 640]
            voxel_grid = torch.rand(self.num_bins, 480, 640)
            # Create a dummy bounding box: [x, y, w, h] (here, fixed size box)
            bbox = torch.tensor([random.uniform(0, 640 - 50), random.uniform(0, 480 - 50), 50, 50])
            samples.append((voxel_grid, bbox, None))
        return samples
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        return self.samples[idx]

#############################################
# Dummy Model and Loss Definitions
#############################################

class BaselineCNN(nn.Module):
    def __init__(self, num_bins, img_height, img_width):
        super(BaselineCNN, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(num_bins, 16, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        flattened = (img_height // 4) * (img_width // 4) * 32
        self.fc = nn.Linear(flattened, 4)  # Predict bounding box [x, y, w, h]
    
    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        bbox = self.fc(x)
        # Dummy confidence
        conf = torch.sigmoid(torch.randn(x.size(0)))
        detection_map = torch.randn(x.size(0), 5, 480, 640)
        return bbox, conf, detection_map

class SpatialDetectionLoss(nn.Module):
    def __init__(self, img_width, img_height, lambda_coord, lambda_noobj):
        super(SpatialDetectionLoss, self).__init__()
        self.img_width = img_width
        self.img_height = img_height
        self.lambda_coord = lambda_coord
        self.lambda_noobj = lambda_noobj
        self.criterion = nn.SmoothL1Loss(reduction='mean')
    
    def forward(self, predictions, targets):
        # predictions: (bbox, conf, detection_map)
        bbox_pred, conf_pred, detection_map = predictions
        loss_bbox = self.criterion(bbox_pred, targets)
        # Dummy components for illustration
        components = {
            'coord_loss': loss_bbox.item(),
            'conf_pos_loss': 0.0,
            'conf_neg_loss': 0.0,
            'direct_conf_loss': 0.0,
            'total_loss': loss_bbox.item()
        }
        return loss_bbox, components

#############################################
# Debug & Evaluation Functions
#############################################

def debug_predictions(model, dataloader, device, num_samples=3):
    """Debug model predictions on a few samples"""
    model.eval()
    examples = []
    
    with torch.no_grad():
        for batch in dataloader:
            if batch is None or len(examples) >= num_samples:
                continue
            voxel_grids, bboxes, _ = batch
            voxel_grids = voxel_grids.to(device)
            bbox_pred, conf_pred, detection_maps = model(voxel_grids)
            for i in range(min(len(voxel_grids), num_samples - len(examples))):
                examples.append({
                    'voxel_grid': voxel_grids[i].cpu(),
                    'gt_bbox': bboxes[i].cpu(),
                    'pred_bbox': bbox_pred[i].cpu(),
                    'confidence': conf_pred[i].item(),
                    'detection_map': detection_maps[i].cpu()
                })
            if len(examples) >= num_samples:
                break
    return examples

def train_one_epoch(model, dataloader, criterion, optimizer, device, clip_grad=None):
    """Train the model for one epoch and return average loss."""
    model.train()
    total_loss = 0
    loss_components = {
        'coord_loss': 0, 
        'conf_pos_loss': 0, 
        'conf_neg_loss': 0, 
        'direct_conf_loss': 0,
        'total_loss': 0
    }
    batches_processed = 0
    
    progress_bar = tqdm(dataloader, desc="Training")
    for i, batch in enumerate(progress_bar):
        if batch is None:
            continue
        voxel_grids, bboxes, _ = batch
        voxel_grids = voxel_grids.to(device)
        bboxes = bboxes.to(device)
        optimizer.zero_grad()
        predictions = model(voxel_grids)
        loss, components = criterion(predictions, bboxes)
        loss.backward()
        if clip_grad is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip_grad)
        optimizer.step()
        total_loss += loss.item()
        batches_processed += 1
        progress_bar.set_postfix({"loss": total_loss / batches_processed})
        if i % 20 == 0:
            with torch.no_grad():
                bbox_pred, conf_pred, _ = predictions
                print(f"\nBatch {i} - Sample 0:")
                print(f"  GT: {bboxes[0].cpu().numpy()}")
                print(f"  Pred: {bbox_pred[0].cpu().numpy()}")
                print(f"  Confidence: {conf_pred[0].item():.4f}")
                print(f"  IoU: {calculate_iou(bbox_pred[0].cpu(), bboxes[0].cpu()):.4f}")
    if batches_processed > 0:
        for k in loss_components:
            loss_components[k] /= batches_processed
    return total_loss / batches_processed if batches_processed > 0 else float('inf'), loss_components

def evaluate(model, dataloader, device, iou_thresholds=[0.5]):
    """Evaluate the model and return metrics."""
    model.eval()
    all_metrics = {f'AP@{thr}': 0 for thr in iou_thresholds}
    all_metrics['Mean_IoU'] = 0
    batches_processed = 0
    debug_samples = []
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            if batch is None:
                continue
            voxel_grids, bboxes, _ = batch
            voxel_grids = voxel_grids.to(device)
            bboxes = bboxes.to(device)
            bbox_pred, conf_pred, _ = model(voxel_grids)
            if len(debug_samples) < 5:
                for i in range(min(2, bbox_pred.size(0))):
                    debug_samples.append({
                        'gt': bboxes[i].cpu().numpy(),
                        'pred': bbox_pred[i].cpu().numpy(),
                        'conf': conf_pred[i].item()
                    })
            metrics = calculate_metrics(bbox_pred, bboxes, iou_thresholds)
            for key, value in metrics.items():
                all_metrics[key] += value
            batches_processed += 1
    print("\n=== Debugging Evaluation Predictions ===")
    for i, sample in enumerate(debug_samples):
        print(f"Sample {i+1} - Ground Truth: {sample['gt']}")
        print(f"Sample {i+1} - Prediction  : {sample['pred']} (Conf: {sample['conf']:.4f})")
    if batches_processed > 0:
        for key in all_metrics:
            all_metrics[key] /= batches_processed
    return all_metrics

def test_iou_calculation():
    """Test the IoU calculation with a known case."""
    box1 = torch.tensor([100.0, 100.0, 50.0, 50.0])
    box2 = torch.tensor([125.0, 125.0, 50.0, 50.0])
    iou = calculate_iou(box1, box2)
    print(f"\n=== Testing IoU Computation ===")
    print(f"Expected IoU ≈ 0.14, Computed IoU: {iou:.4f}")

def run_pretraining_tests(config, train_dataloader, val_dataloader, model):
    print("\n=== Running Pre-training Tests ===")
    test_iou_calculation()
    # Debug predictions on validation set
    print("\n=== Debugging Evaluation Predictions ===")
    for batch in val_dataloader:
        voxel_grids, bboxes, _ = batch
        voxel_grids = voxel_grids.to(config['device'])
        bboxes = bboxes.to(config['device'])
        bbox_pred, _, _ = model(voxel_grids)
        print("Sample - Ground Truth:", bboxes[0].cpu().numpy())
        print("Sample - Prediction  :", bbox_pred[0].detach().cpu().numpy())
        break
    # Visualize a couple of debug samples
    def visualize_samples(num=2):
        samples = 0
        model.eval()
        with torch.no_grad():
            for i, batch in enumerate(val_dataloader):
                voxel_grids, bboxes, _ = batch
                voxel_grids = voxel_grids.to(config['device'])
                bbox_pred, _, detection_maps = model(voxel_grids)
                fig_path = os.path.join(config['save_dir'], f'debug_prediction_{i}.png')
                visualize_voxel_grid(
                    voxel_grids[0].cpu(),
                    labels=[bboxes[0].cpu(), bbox_pred[0].cpu()],
                    save_path=fig_path,
                    detection_map=detection_maps[0].cpu()
                )
                print(f"Saved debug visualization to {fig_path}")
                samples += 1
                if samples >= num:
                    break
    visualize_samples(2)
    print("=== Pre-training Tests Completed ===\n")

#############################################
# Main Training & Evaluation Pipeline
#############################################

def main():
    set_seed(42)
    
    # Configuration
    config = {
        'num_bins': 5,
        'time_window': 10000,
        'batch_size': 8,
        'num_epochs': 15,
        'learning_rate': 1e-3,
        'weight_decay': 1e-5,
        'device': 'cuda' if torch.cuda.is_available() else 'cpu',
        'dataset_path': '../../Datasets/DSEC_Detection/dsec-det/',
        'save_dir': './results',
        'evaluate_every': 1,
        'frame_step': 50,
        'max_samples_per_file': 100,
        'clip_grad': 5.0,
        'cache_dir': './voxel_cache',
        'precompute': True,
        'lambda_coord': 10.0,
        'lambda_noobj': 0.1,
    }
    
    print(f"Using device: {config['device']}")
    os.makedirs(config['save_dir'], exist_ok=True)
    if config['cache_dir']:
        os.makedirs(config['cache_dir'], exist_ok=True)
    
    # Prepare dataset paths
    dataset_path = config['dataset_path']
    train_event_dir = os.path.join(dataset_path, 'train_events/train/')
    train_label_dir = os.path.join(dataset_path, 'train_object_detections/train/')
    
    sequences = [d for d in os.listdir(train_event_dir) if os.path.isdir(os.path.join(train_event_dir, d))]
    print(f"Found {len(sequences)} sequences.")
    
    event_files = []
    label_files = []
    for seq in sequences:
        event_file = os.path.join(train_event_dir, seq, 'events/left/events.h5')
        label_file = os.path.join(train_label_dir, seq, 'object_detections/left/tracks.npy')
        if os.path.exists(event_file) and os.path.exists(label_file):
            event_files.append(event_file)
            label_files.append(label_file)
            print(f"✓ Found: {seq}")
    print(f"Found {len(event_files)} valid sequence pairs.")
    
    print("Loading and processing data...")
    dataset = PrecomputedEventDataset(
        event_files=event_files,
        label_files=label_files,
        num_bins=config['num_bins'],
        time_window=config['time_window'],
        frame_step=config['frame_step'],
        max_samples_per_file=config['max_samples_per_file'],
        precompute=config['precompute'],
        cache_dir=config['cache_dir']
    )
    
    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = random_split(dataset, [train_size, val_size],
                                               generator=torch.Generator().manual_seed(42))
    
    train_dataloader = DataLoader(
        train_dataset,
        batch_size=config['batch_size'],
        shuffle=True,
        num_workers=20,
        pin_memory=True,
        collate_fn=collate_fn,
        persistent_workers=True
    )
    
    val_dataloader = DataLoader(
        val_dataset,
        batch_size=config['batch_size'],
        shuffle=False,
        num_workers=20,
        pin_memory=True,
        collate_fn=collate_fn,
        persistent_workers=True
    )
    
    print(f"Train dataset size: {len(train_dataset)}")
    print(f"Validation dataset size: {len(val_dataset)}")
    
    # Create the model
    model = BaselineCNN(num_bins=config['num_bins'], img_height=480, img_width=640).to(config['device'])
    
    # Run pre-training tests
    run_pretraining_tests(config, train_dataloader, val_dataloader, model)
    
    # Define custom loss function
    criterion = SpatialDetectionLoss(
        img_width=640, 
        img_height=480,
        lambda_coord=config['lambda_coord'],
        lambda_noobj=config['lambda_noobj']
    )
    
    # Create optimizer and learning rate scheduler
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config['learning_rate'],
        weight_decay=config['weight_decay']
    )
    scheduler = torch.optim.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3, verbose=True
    )
    
    best_ap = 0
    best_epoch = -1
    start_time = time.time()
    
    print("\n=== Starting Training ===")
    for epoch in range(config['num_epochs']):
        epoch_start = time.time()
        print(f"\nEpoch {epoch+1}/{config['num_epochs']}")
        
        train_loss, loss_components = train_one_epoch(
            model, train_dataloader, criterion, optimizer, 
            config['device'], clip_grad=config['clip_grad']
        )
        
        epoch_time = time.time() - epoch_start
        print(f"Train Loss: {train_loss:.4f} (Time: {epoch_time:.2f}s)")
        print("Loss Components:")
        for k, v in loss_components.items():
            print(f"  {k}: {v:.4f}")
        
        scheduler.step(train_loss)
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Current learning rate: {current_lr:.6f}")
        
        if (epoch + 1) % config['evaluate_every'] == 0 or epoch == config['num_epochs'] - 1:
            metrics = evaluate(model, val_dataloader, config['device'])
            print("Validation Metrics:")
            for key, value in metrics.items():
                print(f"  {key}: {value:.4f}")
            
            if metrics['AP@0.5'] > best_ap:
                best_ap = metrics['AP@0.5']
                best_epoch = epoch
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'metrics': metrics,
                }, os.path.join(config['save_dir'], 'best_model.pth'))
                print(f"Saved new best model with AP@0.5: {best_ap:.4f}")
            
            if epoch % 5 == 0 or epoch == config['num_epochs'] - 1:
                examples = debug_predictions(model, val_dataloader, config['device'])
                for i, example in enumerate(examples):
                    fig_path = os.path.join(config['save_dir'], f'debug_epoch{epoch+1}_ex{i+1}.png')
                    visualize_voxel_grid(
                        example['voxel_grid'],
                        labels=[example['gt_bbox'], example['pred_bbox']],
                        save_path=fig_path,
                        detection_map=example['detection_map']
                    )
                    print(f"Saved debug visualization to {fig_path}")
    
    total_time = time.time() - start_time
    print(f"\nTraining complete! Total time: {total_time:.2f}s")
    print(f"Best AP@0.5: {best_ap:.4f} at epoch {best_epoch+1}")
    
    # Load best model for final evaluation
    checkpoint = torch.load(os.path.join(config['save_dir'], 'best_model.pth'))
    model.load_state_dict(checkpoint['model_state_dict'])
    
    final_metrics = evaluate(model, val_dataloader, config['device'])
    print("\nFinal Evaluation Metrics:")
    for key, value in final_metrics.items():
        print(f"  {key}: {value:.4f}")
    
    print("\nGenerating final visualizations...")
    examples = debug_predictions(model, val_dataloader, config['device'], num_samples=5)
    for i, example in enumerate(examples):
        fig_path = os.path.join(config['save_dir'], f'final_prediction_{i+1}.png')
        visualize_voxel_grid(
            example['voxel_grid'],
            labels=[example['gt_bbox'], example['pred_bbox']],
            save_path=fig_path,
            detection_map=example['detection_map']
        )
        print(f"Saved visualization to {fig_path}")
        print(f"Example {i+1}:")
        print(f"  Ground Truth: {example['gt_bbox'].numpy()}")
        print(f"  Prediction: {example['pred_bbox'].numpy()}")
        print(f"  Confidence: {example['confidence']:.4f}")
        print(f"  IoU: {calculate_iou(example['pred_bbox'], example['gt_bbox']):.4f}")

if __name__ == "__main__":
    main()